# Chapitre 3 - Lab : les index de PostgreSQL vus de l'intérieur

Ce lab confronte le cours à un vrai moteur : PostgreSQL 16, qui range ses tables dans un
**fichier de tas** et ses index dans des **B-trees** (cours 3 et 4). Nous allons :

1. mesurer un parcours séquentiel et lire un plan d'exécution (`EXPLAIN ANALYZE`) ;
2. inspecter la structure réelle d'un index B-tree (extension `pageinspect`) et observer
   le choix du plan selon la sélectivité ;
3. obtenir un **index-only scan** grâce à un index couvrant (`INCLUDE`) ;
4. mesurer le coût des index en écriture, en octets de **WAL** ;
5. observer la **fragmentation** après des suppressions et le rôle de `VACUUM` ;
6. construire un index **GIN** pour la recherche plein texte.

**Démarrer et arrêter les services** (depuis la racine du dépôt) :

```bash
docker compose -f docker/storage/compose.yml -p ddia_storage up -d     # démarrer
docker compose -f docker/storage/compose.yml -p ddia_storage down -v   # arrêter et effacer
```

PostgreSQL écoute sur le port **10432** de la machine (utilisateur, mot de passe et base :
`ddia` / `ddia` / `stockage`).

In [1]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg

from ddia import storage as S
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, COLORS, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
pd.set_option("display.max_colwidth", 120)

DSN = "host=localhost port=10432 user=ddia password=ddia dbname=stockage"
con = None
for essai in range(30):
    try:
        con = psycopg.connect(DSN, autocommit=True)
        break
    except psycopg.OperationalError:
        time.sleep(1)
assert con is not None, "PostgreSQL ne répond pas : avez-vous lancé docker compose up ?"
print(con.execute("SELECT version()").fetchone()[0][:60])


def q(sql, params=None):
    """Exécute une requête et renvoie toutes les lignes."""
    return con.execute(sql, params).fetchall()


def plan(sql):
    """Plan réel (EXPLAIN ANALYZE, BUFFERS) sous forme de texte."""
    return "\n".join(r[0] for r in q("EXPLAIN (ANALYZE, BUFFERS, TIMING OFF, SUMMARY OFF) " + sql))


def plan_json(sql):
    return q("EXPLAIN (ANALYZE, BUFFERS, FORMAT JSON) " + sql)[0][0][0]

PostgreSQL 16.14 (Debian 16.14-1.pgdg13+1) on x86_64-pc-linu
